In [ ]:
import sys
import torch
from torch import nn
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from utils.util import preprocess_mixed_text

df = pd.read_csv('./data/SMSSpamCollection', 
                    sep="\t", 
                    names=["type", 
                    "message"])

# df.iloc[2]["message"] # 데이터프레임의 특정 행을 확인하는 방법
df["spam"] = df["type"] == "spam"
df.drop("type", axis=1, inplace=True) # inplace: 없애는 열을 복사해두지 말라는 뜻
# df.head()

# documents = [
#     "안녕 내 이름은 이주헌이야.",
#     "만나서 반가워 앞으로 잘 지내보자!",
#     "Hello! world, Today is amazing",
#     "Hello!! mars, today is perpect"
# ]

# 영어+한글 전처리 적용
# processed_documents = [ preprocess_mixed_text(doc) for doc in documents ]
# print("전처리된 문서들:")
# for i, doc in enumerate(processed_documents):
#     print(f"{i}: {doc}")

# 훈련 데이터와 시험 데이터를 분리해야 하는데, 일단 전체 데이터의 훈련데이터를 80% 설정함
df_train = df.sample( frac=0.8, random_state=0 )
df_val = df.drop( index=df_train.index )

cv = CountVectorizer( max_features=1000 )
msg_train = cv.fit_transform( df_train["message"])
msg_val = cv.transform( df_val["message"])
# print( msg[0, :] )
# print( cv.get_feature_names_out()[349])

# test 한글 영어 처리 테스트
# cv.fit( processed_documents )
# print( "\n추출된 특성들: ")
# print( cv.get_feature_names_out() )

# out = cv.transform( processed_documents )
# print( "\n벡터화 결과: ")
# print(out.todense())

# 1. 모델 훈련
X_train = torch.tensor(msg_train.todense(), dtype=torch.float32)
y_train = torch.tensor(df_train["spam"].values, dtype=torch.float32)\
        .reshape((-1, 1))

X_val = torch.tensor(msg_val.todense(), dtype=torch.float32)
y_val = torch.tensor(df_val["spam"].values, dtype=torch.float32)\
        .reshape((-1, 1))

model = nn.Linear( 1000, 1)
# loss_fn = torch.nn.MSELoss() # 평균제곱오차 방식의 손실함수
loss_fn = torch.nn.BCEWithLogitsLoss() # 이진 교차 엔트로피 손실 함수
optimizer = torch.optim.SGD( model.parameters(), lr=0.02 )

for i in range( 0, 10000 ):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = loss_fn( outputs, y_train )
    loss.backward()
    optimizer.step()

    if i % 1000 == 0:
        print( loss )

# 모델의 예측결과가 -가 나와버림 또는 160%가 나오고 있음 > 시그모이드 함수 필요
# But 시그모이드 함수는 현재 평균제곱오차 방식의 MSELoss손실함수를 사용할 경우 대부분 값이 0이 나오게 된다
# 따라서 이 경우 시그모이드 함수를 사용하는 것이 아니라 이진 교차 엔트로피 손실 함수(BCELoss)를 사용하는 것이 좋다

def evaluate_model( X, y):
    model.eval()
    with torch.no_grad():
        y_pred = nn.functional.sigmoid( model(X) ) > 0.5
        print( "accuracy:", (y_pred == y)\
            .type(torch.float32).mean() )


        print( "sensitivity:", (y_pred[ y==1] == y[ y==1 ])\
            .type(torch.float32).mean() )

        print( "specificity:", (y_pred[ y==0] == y[ y==0 ])\
            .type(torch.float32).mean() )

        print( "precision:", (y_pred[ y_pred==1] == y[ y_pred==1 ])\
            .type(torch.float32).mean() )

# print('Evaluating on the training data')
# evaluate_model( X_train, y_train )

# print('Evaluating on the training data')
# evaluate_model( X_val, y_val )


# 모델의 실제 다른 데이터에 사용 예시
custom_messages = cv.transform([
    "We have release a new product, do you want to buy it?",
    "Winner! Great deal, call us to get this product for free",
    "Tommrow is my birthday, do you come to the party?"
])

X_custom = torch.tensor( custom_messages.todense(), dtype=torch.float32 )

model.eval()
with torch.no_grad():
    pred = nn.functional.sigmoid( model( X_custom ))
    print( pred )











tensor(0.6895, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2252, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1641, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1365, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1203, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1093, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1013, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0950, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0900, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.0858, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor([[0.6169],
        [0.0142]])
